# MMGCN · MELD — Colab Reproduction (Baseline)

**Objective:** Leak-free reproduction of [MMGCN](https://arxiv.org/abs/2107.06779) (Hu et al., ACL-IJCNLP 2021) **exclusively on the MELD dataset**. This notebook contains **no learnable cross-modal edge gate, no modality corruption testing, and no architectural modifications.**

- **Official Repository:** https://github.com/hujingwen6666/MMGCN (pinned commit: `85732d984f70c1c84dd47c81aa97f1271397b899`)
- **Dataset & Features:** Uses the official pre-extracted trimodal features `MELD_features/MELD_features_raw1.pkl` (600-d TextCNN, 300-d OpenSMILE audio, 342-d DenseNet visual). Raw audio/video extraction is not re-run.
- **Published Reference Score:** The paper reports **MELD Weighted-F1 = 58.65%**, which is the authors' published benchmark value rather than a measurement from this notebook.
- **Methodological Fixes for Reproducibility:** The official `train.py` uses `valid=0.0` on MELD and selects the peak test score across 60 epochs. Here we introduce a strict dialogue-level 90%/10% train/validation split (`SEED=27350`) and select the checkpoint exclusively via validation Weighted-F1, evaluating the test set only once.
- **64 vs. 4 Layer Discrepancy:** While `run.sh` passes `--Deep_GCN_nlayers 4`, `model.py` hardcodes `nlayers=64` when constructing `MM_GCN`. This notebook wires the parameter to `GRAPH_LAYERS` and defaults to **64** (the actual effective depth in the official repository).

> **Data & Security Note:** Do not commit `.pkl` data files or large `.pt` checkpoints to the public GitHub repository. IEMOCAP requires a restrictive USC license and is not used here.

**Logging:** After every epoch, full Train and Validation metrics are printed live and flushed to `history.csv`.


## 1. Verify Colab GPU and Runtime Versions
In Colab, select **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU). We retain Colab's modern pre-installed Python and PyTorch versions.

In [ ]:
import sys, platform, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA (torch):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Please enable a CUDA GPU: Runtime -> Change runtime type -> T4 GPU"
DEVICE = torch.device("cuda")

## 2. Install Dependencies
We keep Colab's pre-installed PyTorch/CUDA and install modern `torch-geometric` along with scientific Python packages.

In [ ]:
%pip -q install "torch-geometric==2.8.1" "ipdb>=0.13,<1" "scikit-learn>=1.5" "pandas>=2" "matplotlib>=3"

## 3. Clone Official Repository at Pinned Commit
We pin commit `85732d984f70c1c84dd47c81aa97f1271397b899` so upstream changes never affect reproducibility.

In [ ]:
from pathlib import Path
import os, subprocess
ROOT = Path('/content/MMGCN')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/hujingwen6666/MMGCN.git', str(ROOT)], check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','85732d984f70c1c84dd47c81aa97f1271397b899'], check=True)
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'], text=True).strip())
pkl = ROOT/'MELD_features'/'MELD_features_raw1.pkl'
assert pkl.exists(), f'MELD feature file not found: {pkl}'
print('MELD PKL:', round(pkl.stat().st_size / (1024**2), 2), 'MiB')
os.chdir(ROOT)

## 4. Minimal Idempotent Compatibility Patch for Modern PyTorch

This cell is idempotent (safe to re-run). It fixes device placement in `FocalLoss`, explicit `dim=1` in `F.log_softmax`, links `nlayers=64` to `Deep_GCN_nlayers`, and replaces legacy NumPy tuple indexing `adj[idx]` with `adj[idx[0], idx[1]]`. **No architectural or gating changes are introduced.**


In [ ]:
from pathlib import Path

# Idempotent compatibility patch for PyTorch 2.x
model_file = ROOT / 'model.py'
src = model_file.read_text(encoding='utf-8')

old_onehot = 'label_onehot = torch.zeros([seq_length, labels_length]).cuda().scatter_(1, new_label, 1)'
new_onehot = 'label_onehot = torch.zeros([seq_length, labels_length], device=logits.device).scatter_(1, new_label, 1)'
assert old_onehot in src or new_onehot in src, 'FocalLoss onehot line not found'
src = src.replace(old_onehot, new_onehot, 1)

old_softmax = 'log_p = F.log_softmax(logits)'
new_softmax = 'log_p = F.log_softmax(logits, dim=1)'
assert old_softmax in src or new_softmax in src, 'FocalLoss softmax line not found'
src = src.replace(old_softmax, new_softmax, 1)

lines = src.splitlines()
hits = [i for i, line in enumerate(lines) if 'self.graph_model = MM_GCN(' in line]
assert len(hits) == 1, f'Unexpected MMGCN instantiation structure: {hits}'
i = hits[0]
old_layers = 'nlayers=64'
new_layers = 'nlayers=Deep_GCN_nlayers'
if old_layers in lines[i]:
    lines[i] = lines[i].replace(old_layers, new_layers, 1)
    print('MMGCN layer parameter wired to Deep_GCN_nlayers.')
elif new_layers in lines[i]:
    print('MMGCN layer parameter already patched.')
else:
    raise RuntimeError(f'Unexpected MMGCN layer line: {lines[i]}')
model_file.write_text('\n'.join(lines) + '\n', encoding='utf-8')

mm_file = ROOT / 'model_mm.py'
mm_src = mm_file.read_text(encoding='utf-8')
old_idx = 'adj[idx] = dia_sim'
new_idx = 'adj[idx[0], idx[1]] = dia_sim'
assert old_idx in mm_src or new_idx in mm_src, 'Expected edge indexing line not found'
if old_idx in mm_src:
    mm_src = mm_src.replace(old_idx, new_idx, 1)
    mm_file.write_text(mm_src, encoding='utf-8')
    print('MMGCN edge indexing patched.')
else:
    print('MMGCN edge indexing already patched.')
print('Compatibility patches ready.')


## 5. Experiment Configuration
**Quick Check:** Set `SMOKE_TEST=True` to verify a single forward/backward batch. For the full 60-epoch reproduction, set `SMOKE_TEST=False`, `EPOCHS=60`, `GRAPH_LAYERS=64`.

In [ ]:
SEED = 27350                      # Official seed in train.py
GRAPH_LAYERS = 64                 # Effective depth in model.py (overrides run.sh's 4)
EPOCHS = 60                       # MELD run.sh
BATCH_SIZE = 16                   # MELD run.sh
LR = 0.001                        # MELD run.sh
WEIGHT_DECAY = 0.0                # MELD run.sh
DROPOUT = 0.4                     # MELD run.sh
VALID_RATIO = 0.10                # Protocol fix: dialogue-level 10% validation split instead of 0.0
SMOKE_TEST = False                # Set True for a quick 1-batch check; False for full 60-epoch run
SAVE_MODEL = True

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print('Config:', {'layers': GRAPH_LAYERS, 'epochs': EPOCHS, 'smoke_test': SMOKE_TEST, 'seed': SEED})

## 6. Inspect MELD Data and Construct Dialogue-Level Split
The train-test partition is predefined inside the official `.pkl`. We carve out the validation split **strictly at the dialogue level** from the training dialogues; the test set is never used for model selection or hyperparameter tuning.

In [ ]:
from dataloader import MELDDataset
from torch.utils.data import DataLoader, Subset

train_dataset = MELDDataset(str(pkl), train=True)
test_dataset = MELDDataset(str(pkl), train=False)
order = list(range(len(train_dataset)))
random.Random(SEED).shuffle(order)
n_valid = max(1, round(len(order) * VALID_RATIO))
train_indices = order[n_valid:]
valid_indices = order[:n_valid]
assert set(train_indices).isdisjoint(valid_indices)

train_loader = DataLoader(Subset(train_dataset, train_indices), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=train_dataset.collate_fn, num_workers=0)
valid_loader = DataLoader(Subset(train_dataset, valid_indices), batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=train_dataset.collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=test_dataset.collate_fn, num_workers=0)

print('Dialogue counts:', {'train': len(train_indices), 'valid': len(valid_indices), 'test': len(test_dataset)})
first = train_dataset[0]
print('Feature shapes [n_utterances, dim]:', {'text': tuple(first[0].shape),
     'visual': tuple(first[1].shape), 'audio': tuple(first[2].shape)})
print('Speaker matrix:', tuple(first[3].shape), 'label shape:', tuple(first[5].shape))
print('First 10 training dialogue IDs:', [train_dataset.keys[j] for j in train_indices[:10]])

## 7. Initialize Official MMGCN Model
Matches `run.sh` settings (`LSTM` + `MMGCN` + `avl` modalities + speaker embeddings). No new gating or fusion mechanism is added here.

In [ ]:
from model import DialogueGCNModel, FocalLoss

D_text = first[0].shape[-1]
D_visual = first[1].shape[-1]
D_audio = first[2].shape[-1]
n_speakers = first[3].shape[-1]
assert (D_text, D_visual, D_audio) == (600, 342, 300), f'Unexpected MELD feature dims: {(D_text,D_visual,D_audio)}'
assert n_speakers == 9, f'Expected 9 MELD speaker slots, found: {n_speakers}'

model = DialogueGCNModel(
    base_model='LSTM', D_m=D_text, D_g=150, D_p=150, D_e=100, D_h=100,
    D_a=100, graph_hidden_size=100, n_speakers=n_speakers, max_seq_len=200,
    window_past=10, window_future=10, n_classes=7,
    listener_state=False, context_attention='general', dropout=DROPOUT,
    nodal_attention=True, no_cuda=False, graph_type='MMGCN', graph_construct='direct',
    use_residue=True, D_m_v=D_visual, D_m_a=D_audio, modals='avl',
    att_type='concat_subsequently', av_using_lstm=False,
    Deep_GCN_nlayers=GRAPH_LAYERS, dataset='MELD', use_speaker=True, use_modal=False,
).to(DEVICE)
actual_layers = len(model.graph_model.graph_net.convs)
assert actual_layers == GRAPH_LAYERS, (actual_layers, GRAPH_LAYERS)
print('GCN layers:', actual_layers, '/ Total parameters:', sum(p.numel() for p in model.parameters()))
loss_fn = FocalLoss()  # Retains original custom FocalLoss formula
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

## 8. Training and Evaluation Helpers
Checkpoint selection is performed **strictly via validation Weighted-F1**. Once training finishes, the test set is evaluated **once**.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import time, json, csv, copy, math
from pathlib import Path
from datetime import datetime, timezone

OUT = Path('/content/mmgcn_meld_results')
OUT.mkdir(parents=True, exist_ok=True)

def run_epoch(loader, training=False, limit_batches=None):
    model.train(training)
    total_weighted_loss, total_count = 0.0, 0
    all_y, all_pred = [], []
    with torch.set_grad_enabled(training):
        for idx, batch in enumerate(loader):
            textf, visuf, acouf, qmask, umask, labels = [b.to(DEVICE) for b in batch[:-1]]
            lengths = umask.sum(dim=1).long().tolist()
            y = torch.cat([labels[j, :n] for j,n in enumerate(lengths)])
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits, *_ = model(textf, qmask, umask, lengths, acouf, visuf)
            loss = loss_fn(logits, y)
            if not torch.isfinite(loss):
                raise RuntimeError('NaN/Inf loss encountered.')
            if training:
                loss.backward()
                optimizer.step()
            total_weighted_loss += float(loss.detach()) * y.numel()
            total_count += y.numel()
            all_y.extend(y.cpu().tolist())
            all_pred.extend(logits.argmax(dim=1).detach().cpu().tolist())
            if limit_batches is not None and idx + 1 >= limit_batches:
                break
    if total_count == 0:
        raise RuntimeError('Empty dataset loader')
    return {
        'loss': round(total_weighted_loss/total_count, 6),
        'accuracy': round(100*accuracy_score(all_y, all_pred), 4),
        'weighted_f1': round(100*f1_score(all_y, all_pred, average='weighted', zero_division=0), 4),
        'macro_f1': round(100*f1_score(all_y, all_pred, average='macro', zero_division=0), 4),
        'n_utterances': total_count,
        'labels': all_y,
        'predictions': all_pred,
    }

## 9. Optional Smoke Test
When `SMOKE_TEST=True`, one training batch and one validation batch are executed to verify the environment.

In [ ]:
if SMOKE_TEST:
    t0 = time.time()
    one_train = run_epoch(train_loader, training=True, limit_batches=1)
    one_valid = run_epoch(valid_loader, training=False, limit_batches=1)
    print('Smoke test PASSED')
    print('Train 1 batch:', {k:v for k,v in one_train.items() if k not in ('labels','predictions')})
    print('Valid 1 batch:', {k:v for k,v in one_valid.items() if k not in ('labels','predictions')})
    print('Elapsed seconds:', round(time.time()-t0,2))
else:
    print('Smoke test skipped; proceeding to full training.')

## 10. Live Epoch Logging + Best Validation Checkpoint

At the end of every epoch, **Loss, Accuracy, Weighted-F1, and Macro-F1** are printed for both Train and Validation and flushed to `history.csv`. The best checkpoint is selected strictly via **validation Weighted-F1**. Test metrics are not computed during training.


In [ ]:
from tqdm.auto import tqdm

history = []
best_epoch = None
best_valid_f1 = -1.0
best_valid_loss = float('inf')

HISTORY_PATH = OUT / 'history.csv'
HISTORY_FIELDS = [
    'epoch', 'train_loss', 'train_accuracy', 'train_weighted_f1', 'train_macro_f1',
    'valid_loss', 'valid_accuracy', 'valid_weighted_f1', 'valid_macro_f1', 'seconds'
]

if not SMOKE_TEST:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    print(f'Training started: {EPOCHS} epochs | Batch: {BATCH_SIZE} | GPU: {DEVICE}', flush=True)
    print('Per epoch: Train/Valid Loss, Accuracy, Weighted-F1, Macro-F1, elapsed time.', flush=True)

    with open(HISTORY_PATH, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=HISTORY_FIELDS)
        writer.writeheader()
        csvfile.flush()

        for epoch in tqdm(range(1, EPOCHS + 1), desc='Training', unit='epoch'):
            start = time.perf_counter()
            tr = run_epoch(train_loader, training=True)
            va = run_epoch(valid_loader, training=False)
            row = {
                'epoch': epoch,
                'train_loss': tr['loss'],
                'train_accuracy': tr['accuracy'],
                'train_weighted_f1': tr['weighted_f1'],
                'train_macro_f1': tr['macro_f1'],
                'valid_loss': va['loss'],
                'valid_accuracy': va['accuracy'],
                'valid_weighted_f1': va['weighted_f1'],
                'valid_macro_f1': va['macro_f1'],
                'seconds': round(time.perf_counter() - start, 2),
            }
            history.append(row)
            writer.writerow(row)
            csvfile.flush()

            improved = (
                va['weighted_f1'] > best_valid_f1 or
                (va['weighted_f1'] == best_valid_f1 and va['loss'] < best_valid_loss)
            )
            if improved:
                best_valid_f1 = va['weighted_f1']
                best_valid_loss = va['loss']
                best_epoch = epoch
                if SAVE_MODEL:
                    torch.save({
                        'model_state_dict': model.state_dict(),
                        'epoch': epoch,
                        'valid_weighted_f1': best_valid_f1,
                    }, OUT / 'best_mmgcn_meld.pt')
                else:
                    best_state = copy.deepcopy(model.state_dict())

            print(
                f"Epoch [{epoch:02d}/{EPOCHS}] | "
                f"Train: Loss={tr['loss']:.4f}, Acc={tr['accuracy']:.2f}%, "
                f"W-F1={tr['weighted_f1']:.2f}%, Macro-F1={tr['macro_f1']:.2f}% | "
                f"Valid: Loss={va['loss']:.4f}, Acc={va['accuracy']:.2f}%, "
                f"W-F1={va['weighted_f1']:.2f}%, Macro-F1={va['macro_f1']:.2f}% | "
                f"Time={row['seconds']:.1f}s"
                + (' | NEW BEST MODEL' if improved else ''),
                flush=True,
            )

    print(f'Best epoch: {best_epoch} | Validation Weighted-F1: {best_valid_f1:.2f}%', flush=True)
    print(f'Epoch history saved to: {HISTORY_PATH}', flush=True)
else:
    print('SMOKE_TEST=True: Full training skipped.')


## 11. Single Evaluation on the Test Set
Loads the best validation checkpoint and evaluates the MELD test set once. **58.65%** is the authors' published reference score; this cell outputs our measured baseline score.

In [ ]:
if not SMOKE_TEST:
    if SAVE_MODEL:
        checkpoint = torch.load(OUT/'best_mmgcn_meld.pt', map_location=DEVICE, weights_only=True)
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(best_state)
    te = run_epoch(test_loader, training=False)
    print('=== MELD Test Results (Measured Baseline) ===')
    print({k:v for k,v in te.items() if k not in ('labels','predictions')})
    print('Published Reference Weighted-F1: 58.65% (Authors)')
    print('Confusion Matrix (class indices 0..6):')
    print(confusion_matrix(te['labels'], te['predictions'], labels=list(range(7))))
    print(classification_report(te['labels'], te['predictions'], labels=list(range(7)), zero_division=0, digits=4))
else:
    print('TEST SKIPPED (smoke test mode).')

## 12. Save Results and Environment Metadata for the Report
Generated artifacts: `history.csv`, `metrics.json`, `predictions.csv`, `train_validation_curve.png`, `pip_freeze.txt`, `python_torch_versions.json`, and checkpoint (`SAVE_MODEL=True`).

In [ ]:
import subprocess, matplotlib.pyplot as plt
if not SMOKE_TEST:
    with open(OUT/'history.csv','w',newline='') as f:
        writer = csv.DictWriter(f, fieldnames=HISTORY_FIELDS)
        writer.writeheader(); writer.writerows(history)
    with open(OUT/'predictions.csv','w',newline='') as f:
        writer = csv.writer(f); writer.writerow(['true_label_id','predicted_label_id'])
        writer.writerows(zip(te['labels'],te['predictions']))
    source_commit = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'], text=True).strip()
    results = {
        'dataset':'MELD', 'paper_reference_weighted_f1':58.65,
        'source_commit':source_commit, 'seed':SEED, 'validation_ratio':VALID_RATIO,
        'validation_split_unit':'dialogue', 'graph_layers_actual':GRAPH_LAYERS,
        'epochs_requested':EPOCHS, 'best_epoch':best_epoch,
        'batch_size':BATCH_SIZE, 'lr':LR, 'weight_decay':WEIGHT_DECAY,
        'dropout':DROPOUT, 'best_validation_weighted_f1':best_valid_f1,
        'test_weighted_f1':te['weighted_f1'], 'test_macro_f1':te['macro_f1'],
        'test_accuracy':te['accuracy'], 'test_loss':te['loss'],
        'peak_gpu_memory_mib':round(torch.cuda.max_memory_allocated()/(1024**2),1),
        'protocol_deviations':['Train split 10% for validation instead of original zero validation',
                               'Test evaluated only once after validation selection',
                               'Modern torch/PyG and explicit F.log_softmax(dim=1)',
                               'Layer count exposed as config; original effective count is 64',
                               'Original custom FocalLoss retained'],
        'timestamp_utc':datetime.now(timezone.utc).isoformat(),
    }
    (OUT/'metrics.json').write_text(json.dumps(results,indent=2,ensure_ascii=False),encoding='utf-8')
    env = {'python':sys.version, 'torch':torch.__version__, 'cuda_runtime':torch.version.cuda,
           'gpu':torch.cuda.get_device_name(0)}
    (OUT/'python_torch_versions.json').write_text(json.dumps(env,indent=2),encoding='utf-8')
    with open(OUT/'pip_freeze.txt','w') as f:
        subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)
    fig,ax = plt.subplots(figsize=(8,4))
    ax.plot([r['epoch'] for r in history], [r['train_weighted_f1'] for r in history],label='Train Weighted-F1')
    ax.plot([r['epoch'] for r in history], [r['valid_weighted_f1'] for r in history],label='Validation Weighted-F1')
    ax.axvline(best_epoch, linestyle='--', alpha=.5, label=f'Selected epoch {best_epoch}')
    ax.set_xlabel('Epoch');ax.set_ylabel('Weighted-F1 (%)');ax.set_title('MMGCN MELD: Train vs. Validation Weighted-F1')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout();fig.savefig(OUT/'train_validation_curve.png',dpi=150)
    plt.show();plt.close(fig)
    print('Saved files:', sorted(p.name for p in OUT.iterdir()))
else:
    print('SMOKE_TEST mode: report artifacts not written.')

## 13. Download Results as ZIP

In [ ]:
if not SMOKE_TEST:
    import shutil
    archive = shutil.make_archive('/content/MMGCN_MELD_baseline_results', 'zip', root_dir=OUT)
    print('Archive ready:', archive)
    from google.colab import files
    files.download(archive)
else:
    print('Complete full training (SMOKE_TEST=False) to download archive.')